## Section 1.9 – Recursion
### Functions of Functions
Functions are named blocks of code. You can write any code in a function, including calls to other functions.

In [1]:
def function_one(x):
    if x > 0:
        return function_two(x)
    else:
        return x * 2
    
def function_two(x):
    if x < 0:
        return function_one(x)
    else:
        return x * 3
    
function_one(10)

30

Of course this should not be news to you. We've been using functions like `len` inside our own functions since we first introduced them.

### Recursion
But here's a weird thing: functions can also call *themselves*. This is called **recursion**. Here is a really trivial example of a recursive function that implements `abs(x)` (note it is *not* a good way to implement this function, just a demonstration).

In [ ]:
def recursion(x):
    if x < 0:
        return recursion(-x) # When it is negative, we call the function again with the value of `x` multiplied by `-1`
    else:
        return x
    
recursion(-10)

10

Read that code carefully. When the input for `x` is positive it is just returned unchanged. When it is negative, we call the function again with the value of `x` multiplied by `-1` (this is what writing `-x` does). This will be a positive value, so it will be returned unchanged, meaning we get the positive value as the final return value.

Of course, we could have just written `return -x` instead, and this would be one less function call – I did say it is not a good way to implement the function. Make sure you understand the difference between the two versions.

Here's a slightly more sophisticated example which calculates the *factorial* of a number. The factorial of $n$ is calculated by taking the product (multiplication) of all positive integers from $n$ to $1$. So five factorial, written $5!$ is calculated like this $$5! = 5 \times 4 \times 3 \times 2 \times 1 = 120$$

In [3]:
def factorial(n):
    if n == 1:
        return 1
    else:
        return n * factorial(n-1)
    
factorial(5)

120

What's happening here? Well, when we call `factorial(5)`, we check if `n` is equal to `1`, which it isn't, so we get to the line: `return n * factorial(n-1)`. This is where all that practice evaluating expressions comes in! We can't evaluate the `*` until we have a value for the right hand side. So first, we must evaluate `factorial(n-1)`, i.e. `factorial(4)`. So, we put aside our value of `n`, and go call `factorial(4)`...

Inside `factorial(4)` we have the same dilemma. We need to know the value of `factorial(3)`, so we put aside our value of `n` and call `factorial(3)`. But `factorial(3)` requires `factorial(2)`, which requires `factorial(1)`... finally we have a return value! `factorial(1)` returns `1`. So now the code goes back through all of previous function calls:
* `factorial(2)` returns `2 * 1`, which is `2`
* `factorial(3)` returns `3 * 2`, which is `6`
* `factorial(4)` returns `4 * 6`, which is `24`
* `factorial(5)` returns `5 * 24`, which is `120`

And we are done.

Recursion is a lot like iteration. We could have written the factorial function like this:

In [4]:
def factorial_itr(n):
    total = 1
    # remember that range(a, b) goes from a to b-1
    for i in range(2, n+1):
        total *= i
    return total

factorial_itr(5)

120

Recursion is an alternative to using a loop. Anything written with recursion can be written with iteration, and vice versa. But some code really lends itself to being written recursively or iteratively. When using Python, you should probably lean towards using a loop as your first option – every time you make a recursive call Python has to remember what line of code to go back to and any local variables, so it is slightly less efficient.

Some languages are built around the idea of recursive functions and have limited or no support for loops. These are called *functional* programming languages. We will learn some concepts from functional programming that we can use in Python in a later chapter.

Despite being slightly less efficient, some algorithms really are much easier to understand using recursion, and actually quite difficult to convert into a version that just uses loops. This is due to the way we can pass values “between” the function calls as parameters and return values, but each function will remember its own local variables. 

So, recursion is there to help! It is worth really trying to get your head around how to write recursive functions. It will come in handy when we need it later.

### Another Example
Another common demonstration of the power of recursion is the *Fibonacci* sequence. This sequence is defined like this:
$$\begin{align} 
    f(0) &= 0 \\
    f(1) &= 1 \\
    f(n) &= f(n-1) + f(n-2) \text{ for all } n > 1
\end{align}$$

Each value after $0$ and $1$ are found by summing the previous two values. So the third value is $0 + 1 = 1$, then $1 + 1 = 2$, then $1 + 2 = 3$, then $2 + 3 = 5$, and so on.

The sequence is famous for having lots of interesting properties, you can read more [on Wikipedia](https://en.wikipedia.org/wiki/Fibonacci_number).

To write a recursive function, we generally want to follow a pattern that looks something like this:
```python
def recursive(x):
    if <base case>:
        return <some actual value>
    else:
        return ... recursive(<move towards base case>)
```

This is the pattern we used to write `factorial` above. The base case was that when `n` equals `1`, we know the return value should be `1`. Then whenever we make the recursive call, we move towards the base case – for `factorial` that means making the number one smaller, so we know we will eventually reach `1` for all positive integers.

Here's a recursive version of a function which calculates the nth Fibonacci number:

In [3]:
def fibonacci_v1(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci_v1(n-1) + fibonacci_v1(n-2)
    
fibonacci_v1(7)

13

This function has two base cases, one for `0` and one for `1`.

The function works, but even reasonable values of `n` cause it to run quite slowly. Run the line below – you will likely see it taking a while to run. If it takes longer than a few seconds then stop the execution and try again with a smaller value – you do not have all day!

In [9]:
fibonacci_v1(30)

832040

Go much larger than this and the function will actually cause an error: there is a limit on how many recursive calls a function is allowed to make. 

The problem is that every time the function is called it makes two function calls. Each of those makes two function calls, and so on. This is very inefficient.
```
               f(4)
              /    \
             /      \
            /        \
         f(3)         f(2)
        /    \       /    \
       /      \   f(1)    f(0)
    f(2)      f(1) |       |
   /    \      |   1       0
f(1)    f(0)   1
 |       |
 1       0
```
Now imagine what the tree for `f(30)` would look like.

An iterative version of Fibonacci would not have this problem. But we can code an efficient recursive version too, and it even lets us demonstrate a nice Python feature: we can include **default values** for our function parameters. Have a look:

In [12]:
def fibonacci_v2(n, last_sum=1, sum_so_far=0):
    if n == 0:
        return sum_so_far
    else:
        return fibonacci_v2(n-1, sum_so_far, last_sum + sum_so_far)
    
fibonacci_v2(6)

8

Take some time to read this code. Try writing out some examples on paper to see how it works. It is much more efficient, allowing us to compute far larger Fibonacci numbers relatively quickly:

Still, Python is not optimised for recursion the way that many functional languages are. So if you keep increasing the value for `n`, you will eventually hit the maximum recursion depth and get an error.

### Questions
#### Question 1: Reverse String
Write a recursive function that reverses an input string. *Hint: follow the general pattern above, and use the length of the string as the basis for the base case.*

In [1]:
%run ../scripts/show_examples.py ./questions/1.9/reverse

Example tests for function reverse

Test 1/5: reverse('hello') -> 'olleh'
Test 2/5: reverse('zebra') -> 'arbez'
Test 3/5: reverse('fiddledeedee') -> 'eedeedelddif'
Test 4/5: reverse('balderdash') -> 'hsadredlab'
Test 5/5: reverse('recursion is more fun than looping') -> 'gnipool naht nuf erom si noisrucer'


In [18]:
def reverse(text):
    if len(text) == 0:
        return text
    else:
        return reverse(text[1:]) +text[0]

print(reverse('ash'))
    
%run -i ../scripts/function_tester.py ./questions/1.9/reverse

hsa
Running tests on function reverse

Test 1/7: 
	inputs: 'hello'
	expected: 'olleh'
	actual: 'olleh'
	result: PASS
Test 2/7: 
	inputs: 'zebra'
	expected: 'arbez'
	actual: 'arbez'
	result: PASS
Test 3/7: 
	inputs: 'fiddledeedee'
	expected: 'eedeedelddif'
	actual: 'eedeedelddif'
	result: PASS
Test 4/7: 
	inputs: 'balderdash'
	expected: 'hsadredlab'
	actual: 'hsadredlab'
	result: PASS
Test 5/7: 
	inputs: 'recursion is more fun than looping'
	expected: 'gnipool naht nuf erom si noisrucer'
	actual: 'gnipool naht nuf erom si noisrucer'
	result: PASS
Test 6/7: 
	inputs: '#?*!'
	expected: '!*?#'
	actual: '!*?#'
	result: PASS
Test 7/7: 
	inputs: ''
	expected: ''
	actual: ''
	result: PASS

Running 3 secret tests...

Test 1/3: PASS
Test 2/3: PASS
Test 3/3: PASS

All tests passed! Great job!


#### Question 2: Recursive Collatz Conjecture
Last section we looked at an interesting mathematical sequence which always seems to end up resulting in a $1$, but this has not been proven. Writing it mathematically, the sequence is formed by successive applications of the following function:

$$f(n) = \begin{cases} 
      3n+1 & n \text{ is odd} \\
      \frac{n}{2} & n \text{ is even} 
   \end{cases}$$
   
The Collatz conjecture asks: for any positive integer $n$, is $f^i(n)=1$ for some $i$?

For this question, write the simulation of the Collatz conjecture using recursion instead of iteration. Return the number of steps required to reach $1$. 

You may assume that the input `n` is greater than or equal to `1`.

In [3]:
%run ../scripts/show_examples.py ./questions/1.9/collatz

Example tests for function collatz

Test 1/5: collatz(2) -> 1
Test 2/5: collatz(3) -> 7
Test 3/5: collatz(4) -> 2
Test 4/5: collatz(5) -> 5
Test 5/5: collatz(6) -> 8


In [ ]:
# Correct answer - option 1:
def collatz(n, steps=0):
    if n == 1:
        return steps
    else:
        if n % 2 == 0:
            return collatz(n/2, steps +1) # we can get rid of the steps += 1 and add this directly in the parameter
        else:
            return collatz((3*n)+1, steps +1)  # we can get rid of the steps += 1 and add this directly in the parameter

collatz(3)
    
%run -i ../scripts/function_tester.py ./questions/1.9/collatz

Running tests on function collatz

Test 1/10: 
	inputs: 2
	expected: 1
	actual: 1
	result: PASS
Test 2/10: 
	inputs: 3
	expected: 7
	actual: 7
	result: PASS
Test 3/10: 
	inputs: 4
	expected: 2
	actual: 2
	result: PASS
Test 4/10: 
	inputs: 5
	expected: 5
	actual: 5
	result: PASS
Test 5/10: 
	inputs: 6
	expected: 8
	actual: 8
	result: PASS
Test 6/10: 
	inputs: 12
	expected: 9
	actual: 9
	result: PASS
Test 7/10: 
	inputs: 637
	expected: 56
	actual: 56
	result: PASS
Test 8/10: 
	inputs: 516
	expected: 123
	actual: 123
	result: PASS
Test 9/10: 
	inputs: 858
	expected: 103
	actual: 103
	result: PASS
Test 10/10: 
	inputs: 899
	expected: 116
	actual: 116
	result: PASS

Running 5 secret tests...

Test 1/5: PASS
Test 2/5: PASS
Test 3/5: PASS
Test 4/5: PASS
Test 5/5: PASS

All tests passed! Great job!


In [ ]:
# Correct answer - option 2:
def collatz(n):
    if n == 1:
        return 0
    else:
        if n % 2 == 0:
            return 1 + collatz(n/2)   # by just adding 1 we are counting and when we get to base case we just return 0
        else:
            return 1 + collatz((3*n)+1)  

collatz(4)

# collatz(1)   = 0
# collatz(2)   = 1 + 0 = 1
# collatz(4)   = 1 + 1 = 2
    
%run -i ../scripts/function_tester.py ./questions/1.9/collatz

Running tests on function collatz

Test 1/10: 
	inputs: 2
	expected: 1
	actual: 1
	result: PASS
Test 2/10: 
	inputs: 3
	expected: 7
	actual: 7
	result: PASS
Test 3/10: 
	inputs: 4
	expected: 2
	actual: 2
	result: PASS
Test 4/10: 
	inputs: 5
	expected: 5
	actual: 5
	result: PASS
Test 5/10: 
	inputs: 6
	expected: 8
	actual: 8
	result: PASS
Test 6/10: 
	inputs: 758
	expected: 59
	actual: 59
	result: PASS
Test 7/10: 
	inputs: 344
	expected: 32
	actual: 32
	result: PASS
Test 8/10: 
	inputs: 731
	expected: 139
	actual: 139
	result: PASS
Test 9/10: 
	inputs: 902
	expected: 54
	actual: 54
	result: PASS
Test 10/10: 
	inputs: 75
	expected: 14
	actual: 14
	result: PASS

Running 5 secret tests...

Test 1/5: PASS
Test 2/5: PASS
Test 3/5: PASS
Test 4/5: PASS
Test 5/5: PASS

All tests passed! Great job!


In [ ]:
# Leaving as a back example fo what not to
def collatz(n):
    steps = 1 ### Do not do this as then each time we run the function again steps rest
    if n == 1:
        return steps -1
    else:
        if n % 2 == 0:
            steps +=1.  # you can also get rid of this if you add steps in as a parameter
            return collatz(n/2)
        else:
            steps +=1 # you can also get rid of this if you add steps in as a parameter
            return collatz((3*n)+1) 

collatz(3)
    
%run -i ../scripts/function_tester.py ./questions/1.9/collatz

Running tests on function collatz

Test 1/10: 
	inputs: 2
	expected: 1
	actual: 0
	result: FAIL

Try editing your code and re-running the cell.


Think about the benefits and trade-offs of writing the Collatz function recursively. One trade-off is the maximum recursion depth, which isn't a problem with iteration. However, the way you end up writing the recursive version has an elegance to it – it is closer to the way we would write the function mathematically. For some problems a recursive solution is easier to implement for this reason – sometimes greatly so. So, do not discard recursion just because iteration is conceptually easier. It is an important tool to have in your toolbox.

## What Next?
When you are done with this notebook, go back to Engage to read the wrap up for week 1.